# FocusRoom — 01: Data Exploration
**Goal:** Understand what the raw dataset looks like before touching a single model weight.

This notebook answers:
- How many images per class?
- Are the images actually 48×48 grayscale, or do we need to resize?
- Is there class imbalance? How bad?
- Do the images look correct (are labels sane)?
- What augmentation strategy should we use?

Run this **before** `02_model_training.ipynb`.

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from collections import Counter
import cv2

# ── Config ──────────────────────────────────────────
RAW_DIR       = '../data/raw'          # focused/ distracted/ closed/
PROCESSED_DIR = '../data/processed'   # train/ val/ test/ splits go here
CLASSES       = ['focused', 'distracted', 'closed']
SEED          = 42
random.seed(SEED)
np.random.seed(SEED)
print('Libraries loaded ✓')

## 1. Class Distribution

In [ ]:
# Count images per class in raw/
class_counts = {}
for cls in CLASSES:
    cls_dir = os.path.join(RAW_DIR, cls)
    if not os.path.exists(cls_dir):
        print(f'  WARNING: {cls_dir} not found — did you download the datasets?')
        class_counts[cls] = 0
        continue
    files = [f for f in os.listdir(cls_dir)
              if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    class_counts[cls] = len(files)
    print(f'  {cls:12s}: {len(files):,} images')

total = sum(class_counts.values())
print(f'\n  Total images : {total:,}')

# Class balance check
print('\n  Class proportions:')
for cls, count in class_counts.items():
    pct = (count / total * 100) if total > 0 else 0
    bar = '█' * int(pct / 2)
    print(f'  {cls:12s}: {pct:5.1f}%  {bar}')

In [ ]:
# Bar chart of class distribution
fig, ax = plt.subplots(figsize=(7, 4))
colours = ['#2D6A4F', '#E76F51', '#1A1A2E']
bars = ax.bar(CLASSES, [class_counts[c] for c in CLASSES], color=colours, width=0.5, edgecolor='white')
ax.set_title('Class Distribution — Raw Dataset', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Image Count')
ax.set_xlabel('Eye State Class')
for bar, cls in zip(bars, CLASSES):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 200,
            f'{class_counts[cls]:,}', ha='center', va='bottom', fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('../data/class_distribution.png', dpi=120)
plt.show()
print('Saved → data/class_distribution.png')

## 2. Sample Image Grid — Visual Sanity Check

In [ ]:
def load_random_samples(class_dir, n=8):
    files = [f for f in os.listdir(class_dir)
             if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    chosen = random.sample(files, min(n, len(files)))
    images = []
    for fname in chosen:
        img = Image.open(os.path.join(class_dir, fname)).convert('L')  # grayscale
        images.append(np.array(img))
    return images

n_per_class = 6
fig, axes = plt.subplots(len(CLASSES), n_per_class, figsize=(14, 6))
fig.suptitle('Random Sample Images per Class', fontsize=14, fontweight='bold', y=1.01)

for row, cls in enumerate(CLASSES):
    cls_dir = os.path.join(RAW_DIR, cls)
    if not os.path.exists(cls_dir):
        continue
    samples = load_random_samples(cls_dir, n=n_per_class)
    for col, img in enumerate(samples):
        ax = axes[row][col]
        ax.imshow(img, cmap='gray', vmin=0, vmax=255)
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(cls, fontsize=11, fontweight='bold',
                          color=colours[row], rotation=0, labelpad=50, va='center')

plt.tight_layout()
plt.savefig('../data/sample_grid.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → data/sample_grid.png')
print('\nLook at this grid carefully:')
print('  ✓ focused   — eyes clearly open, iris visible, gaze roughly forward')
print('  ✓ distracted — eyes open but gaze shifted left/right/up')
print('  ✓ closed    — eyelids closed or mostly closed')
print('  If any images look wrong — mislabelled, corrupted, or non-eye — remove them now.')

## 3. Image Size & Format Audit

In [ ]:
# Check what sizes exist in the raw data — we need to know how much
# resizing prepare_dataset.py will have to do
size_counter = Counter()
corrupt_files = []
total_checked = 0
TARGET_SIZE = (48, 48)

for cls in CLASSES:
    cls_dir = os.path.join(RAW_DIR, cls)
    if not os.path.exists(cls_dir):
        continue
    for fname in os.listdir(cls_dir):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            continue
        fpath = os.path.join(cls_dir, fname)
        try:
            img = Image.open(fpath)
            size_counter[img.size] += 1
            total_checked += 1
        except Exception as e:
            corrupt_files.append((fpath, str(e)))

print(f'  Checked {total_checked:,} images\n')
print(f'  Top image sizes found:')
for size, count in size_counter.most_common(10):
    already_correct = '✓ already target size' if size == TARGET_SIZE else ''
    print(f'    {str(size):15s}: {count:,} images  {already_correct}')

already_correct = size_counter.get(TARGET_SIZE, 0)
needs_resize = total_checked - already_correct
print(f'\n  Already {TARGET_SIZE}: {already_correct:,} ({already_correct/total_checked*100:.1f}%)')
print(f'  Needs resize  : {needs_resize:,} ({needs_resize/total_checked*100:.1f}%)')

if corrupt_files:
    print(f'\n  ⚠ Corrupt/unreadable files: {len(corrupt_files)}')
    for fpath, err in corrupt_files[:5]:
        print(f'    {fpath}: {err}')
else:
    print('\n  ✓ No corrupt files found')

## 4. Pixel Intensity Distribution — Lighting Analysis

In [ ]:
# Sample 100 images per class and plot pixel intensity histograms.
# This tells us if images are too dark (low lighting is our #1 accuracy risk)
# and informs augmentation brightness_range values in train.py

N_SAMPLE = 100
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
fig.suptitle('Pixel Intensity Distribution per Class\n(peak at low values = dark images; peak at high = bright)', fontsize=12)

for i, (cls, colour) in enumerate(zip(CLASSES, colours)):
    cls_dir = os.path.join(RAW_DIR, cls)
    if not os.path.exists(cls_dir):
        continue
    files = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    sample = random.sample(files, min(N_SAMPLE, len(files)))
    all_pixels = []
    for fname in sample:
        arr = np.array(Image.open(os.path.join(cls_dir, fname)).convert('L'))
        all_pixels.extend(arr.flatten().tolist())

    mean_px = np.mean(all_pixels)
    axes[i].hist(all_pixels, bins=64, color=colour, alpha=0.8, density=True)
    axes[i].axvline(mean_px, color='red', linestyle='--', linewidth=1.5, label=f'mean={mean_px:.0f}')
    axes[i].set_title(cls, fontweight='bold')
    axes[i].set_xlabel('Pixel value (0=black, 255=white)')
    axes[i].legend()
    axes[i].spines[['top','right']].set_visible(False)

axes[0].set_ylabel('Density')
plt.tight_layout()
plt.savefig('../data/intensity_distribution.png', dpi=120)
plt.show()
print('\nInterpretation guide:')
print('  mean < 60  → very dark dataset, train.py brightness augmentation is critical')
print('  mean 60-180 → normal range, default augmentation settings are fine')
print('  mean > 180 → very bright, reduce brightness_range upper bound in train.py')

## 5. Class Weight Recommendation

In [ ]:
# Compute and explain the class weights that train.py uses.
# MRL dataset has many more 'closed' images — without weights,
# the model will become biased toward predicting 'closed'.

counts = np.array([class_counts[c] for c in CLASSES], dtype=float)
total  = counts.sum()
weights = total / (len(CLASSES) * counts)

print('  Class weights for model.fit(class_weight=...):')
print('  ' + '─'*40)
for i, (cls, w) in enumerate(zip(CLASSES, weights)):
    print(f'  {i}: {cls:12s}  weight = {w:.4f}')

print('\n  class_weight = {')
for i, w in enumerate(weights):
    print(f'    {i}: {w:.4f},')
print('  }')
print('\n  How to read: a weight of 2.0 means that class is treated as')
print('  twice as important — its loss contributes more to gradient updates.')
print('  This directly counteracts imbalance without discarding images.')

## 6. Augmentation Preview

In [ ]:
# Show what train.py's augmentation pipeline actually does to an image.
# Pick one image and apply all augmentations manually.
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, array_to_img, load_img

# Pick a focused image to demonstrate
focused_dir = os.path.join(RAW_DIR, 'focused')
if os.path.exists(focused_dir):
    sample_file = random.choice([f for f in os.listdir(focused_dir)
                                  if f.lower().endswith(('.jpg','.jpeg','.png'))])
    sample_path = os.path.join(focused_dir, sample_file)

    orig = load_img(sample_path, color_mode='grayscale', target_size=(48, 48))
    orig_arr = img_to_array(orig)          # shape: (48, 48, 1)
    orig_batch = np.expand_dims(orig_arr, 0)

    aug = ImageDataGenerator(
        rotation_range=10,
        width_shift_range=0.1,
        height_shift_range=0.1,
        zoom_range=0.1,
        horizontal_flip=True,
        brightness_range=[0.6, 1.4],
        fill_mode='nearest',
    )

    n_aug = 8
    fig, axes = plt.subplots(1, n_aug + 1, figsize=(14, 2.5))
    axes[0].imshow(orig_arr[:,:,0], cmap='gray', vmin=0, vmax=255)
    axes[0].set_title('Original', fontweight='bold')
    axes[0].axis('off')

    aug_iter = aug.flow(orig_batch, batch_size=1)
    for j in range(1, n_aug + 1):
        aug_img = next(aug_iter)[0].astype('uint8')
        axes[j].imshow(aug_img[:,:,0], cmap='gray', vmin=0, vmax=255)
        axes[j].set_title(f'Aug {j}', fontsize=9)
        axes[j].axis('off')

    fig.suptitle('Augmentation Preview — same image, 8 different random transforms', fontsize=11)
    plt.tight_layout()
    plt.savefig('../data/augmentation_preview.png', dpi=120)
    plt.show()
    print('Saved → data/augmentation_preview.png')
else:
    print('focused/ directory not found — skipping augmentation preview.')

## 7. Summary & Next Steps

In [ ]:
print('═'*55)
print('  EXPLORATION SUMMARY')
print('═'*55)
print(f'  Total images     : {total:,}')
for cls in CLASSES:
    print(f'  {cls:12s}   : {class_counts[cls]:,}')
print()
print('  Saved charts:')
print('    data/class_distribution.png')
print('    data/sample_grid.png')
print('    data/intensity_distribution.png')
print('    data/augmentation_preview.png')
print()
print('  NEXT STEP → run scripts/prepare_dataset.py to create train/val/test splits')
print('  THEN      → open 02_model_training.ipynb')
print('═'*55)